In [0]:
cubhub_claims = dbutils.widgets.get("cubhub_claims")
cubhub_charges = dbutils.widgets.get("cubhub_charges")
cubhub_transactions = dbutils.widgets.get("cubhub_transactions")

billingledger_tbl = dbutils.widgets.get("billingledger_tbl")
billingledgerdetail_tbl = dbutils.widgets.get("billingledgerdetail_tbl")
patientpayer_tbl = dbutils.widgets.get("patientpayer_tbl")
patient_tbl = dbutils.widgets.get("patient_tbl")
branch_tbl = dbutils.widgets.get("branch_tbl")
payer_tbl = dbutils.widgets.get("payer_tbl")
financialclass_tbl = dbutils.widgets.get("financialclass_tbl")
person_tbl = dbutils.widgets.get("person_tbl")
office_tbl = dbutils.widgets.get("office_tbl")
admission_tbl = dbutils.widgets.get("admission_tbl")
patientcareteam_tbl = dbutils.widgets.get("patientcareteam_tbl")
physician_tbl = dbutils.widgets.get("physician_tbl")
physicianperson_tbl = dbutils.widgets.get("physicianperson_tbl")
personaddress_tbl = dbutils.widgets.get("personaddress_tbl")
address_tbl = dbutils.widgets.get("address_tbl")
personphonenumber_tbl = dbutils.widgets.get("personphonenumber_tbl")
phonenumber_tbl = dbutils.widgets.get("phonenumber_tbl")
payeraddress_tbl = dbutils.widgets.get("payeraddress_tbl")
payerbillingcode_tbl = dbutils.widgets.get("payerbillingcode_tbl")
icd_tbl = dbutils.widgets.get("icd_tbl")
payment_tbl = dbutils.widgets.get("payment_tbl")
writeoff_tbl = dbutils.widgets.get("writeoff_tbl")
assignment_tbl = dbutils.widgets.get("assignment_tbl")
assignmentbilling_tbl = dbutils.widgets.get("assignmentbilling_tbl")
payerauthorization_tbl = dbutils.widgets.get("payerauthorization_tbl")
payerauthorizationsegmentbillingcode_tbl = dbutils.widgets.get("payerauthorizationsegmentbillingcode_tbl")
billingcode_tbl = dbutils.widgets.get("billingcode_tbl")
payerbillingcodemodifier_tbl = dbutils.widgets.get("payerbillingcodemodifier_tbl")
billingcodemodifier_tbl = dbutils.widgets.get("billingcodemodifier_tbl")
writeoffline_tbl = dbutils.widgets.get("writeoffline_tbl")
paymentline_tbl = dbutils.widgets.get("paymentline_tbl")

In [0]:
spark.sql(f"""
TRUNCATE TABLE {cubhub_claims}
""")

In [0]:
display(
spark.sql(f"""
INSERT INTO {cubhub_claims} 
SELECT DISTINCT
    b.Id AS OfficeId,
    bl.Id AS ClaimId,
    TRIM(SPLIT(bl.DatesOfService, '-')[0]) AS ServiceStart,
    TRIM(SPLIT(bl.DatesOfService, '-')[1]) AS ServiceEnd,
    DATE_FORMAT(a.AdmitDate, 'MM/dd/yyyy') AS AdmitDate,
    NULL AS DischargeDate,
    p.MedicalRecordNumber,
    addr.Street1,
    addr.Street2,
    addr.City,
    addr.State,
    addr.ZipCode,
    DATE_FORMAT(per.DateOfBirth, 'MM/dd/yyyy') AS DateOfBirth,
    per.FirstName,
    per.LastName,
    per.MiddleName,
    CASE per.Gender 
        WHEN 1 THEN 'Male'
        WHEN 2 THEN 'Female'
        ELSE 'Unknown'
    END AS Gender,
    CASE 
        WHEN pn.Number IS NOT NULL AND LENGTH(pn.Number) = 10 
        THEN CONCAT('(', SUBSTRING(pn.Number, 1, 3), ') ', SUBSTRING(pn.Number, 4, 3), '-', SUBSTRING(pn.Number, 7, 4))
        ELSE pn.Number
    END AS PhoneNumber,
    per.SocialSecurityNumber,
    ph.Npi AS OrderingPhysicianNpi,
    phPer.FirstName AS OrderingPhysicianFirstName,
    phPer.LastName AS OrderingPhysicianLastName,
    ph.Id AS OrderingPhysicianId,
    NULL AS PrimaryPayerSubscriberId,
    pp.PayerIdNumber AS ClaimSubscriberId,
    bl.AmountBilled - COALESCE(pay_agg.TotalPayments, 0) AS Balance,
    bl.ClaimNumber,
    CASE 
        WHEN evv_agg.HasSandata = 1 AND evv_agg.HasCellTrak = 1 THEN 'CellTrak Sandata'
        WHEN evv_agg.HasCellTrak = 1 THEN 'CellTrak'
        WHEN evv_agg.HasSandata = 1 THEN 'Sandata'
        ELSE 'None'
    END AS EvvAggregator,
    bl.AmountBilled AS ExpectedTotal,
    DATE_FORMAT(bl.DateBilled, 'MM/dd/yyyy') AS DateBilled,
    0 AS Adjustments, 
    COALESCE(pay_agg.TotalPayments, 0) AS Payments,
    bl.AmountBilled AS TotalBilled,
    ofc.NationalProviderIdentifier AS Npi,
    'Weekly' AS BillingPeriod,
    payAddr.Street1 AS PayerStreet1,
    payAddr.Street2 AS PayerStreet2,
    payAddr.City AS PayerCity,
    payAddr.State AS PayerState,
    payAddr.ZipCode AS PayerZipCode,
    pay.Name AS PayerName,
    pay.Id AS PayerId,
    CASE p.MedicaidProgram
        WHEN 0 THEN 'Not Selected'
        ELSE CAST(p.MedicaidProgram AS STRING)
    END AS MedicaidProgram,
    NULL AS CobBenefitDate,
    CASE pp.CobDenialReason
        WHEN 0 THEN 'Not Selected'
        ELSE CAST(pp.CobDenialReason AS STRING)
    END AS CobDenialReason,
    CAST(pp.CobEffectiveStartDate AS STRING) AS CobEffectiveDate,
    CASE pp.CobStatus
        WHEN 0 THEN 'Not Selected'
        ELSE CAST(pp.CobStatus AS STRING)
    END AS CobLevel,
    NULL AS CobPrimaryInsurance,
    CASE 
        WHEN pp.SplitDecisionDetails IS NULL THEN 'Not Selected'
        ELSE pp.SplitDecisionDetails
    END AS CobSplitDecision,
    NULL AS CobStatus,
    fc.Abbreviation AS CobType,
    fc.DisplayName AS FinancialClassAbbreviation,
    b.ExternalId AS FinancialClass,
    icd.Icd9Cm AS OfficeExternalId,
    NULL AS PrimaryDiagnosis,
    CURRENT_DATE() AS LoadDate,
    CAST(CURRENT_DATE() AS STRING) AS FileName
FROM {billingledger_tbl} bl
LEFT JOIN {billingledgerdetail_tbl} bld ON bl.Id = bld.BillingLedgerId
JOIN {patientpayer_tbl} pp ON pp.Id = bl.PatientPayerId
JOIN {patient_tbl} p ON p.Id = pp.PatientId
JOIN {branch_tbl} b ON b.Id = p.BranchId
JOIN {payer_tbl} pay ON pay.Id = pp.PayerId
LEFT JOIN {financialclass_tbl} fc ON fc.Id = pay.FinancialClassId
JOIN {person_tbl} per ON per.Id = p.PersonId
LEFT JOIN {office_tbl} ofc ON CAST(b.ExternalId AS INT) = ofc.OfficeNumber
LEFT JOIN (
    SELECT PatientId, AdmitDate,
           ROW_NUMBER() OVER (PARTITION BY PatientId ORDER BY AdmitDate DESC) as rn
    FROM {admission_tbl}
) a ON a.PatientId = p.Id AND a.rn = 1
LEFT JOIN (
    SELECT PatientId, PhysicianId,
           ROW_NUMBER() OVER (PARTITION BY PatientId ORDER BY Id DESC) as rn
    FROM {patientcareteam_tbl}
    WHERE PhysicianId IS NOT NULL
) pct ON pct.PatientId = p.Id AND pct.rn = 1
LEFT JOIN {physician_tbl} ph ON ph.Id = pct.PhysicianId
LEFT JOIN {physicianperson_tbl} phPer ON phPer.Id = ph.PersonId
LEFT JOIN (
    SELECT PersonId, AddressId,
           ROW_NUMBER() OVER (PARTITION BY PersonId ORDER BY Id) as rn
    FROM {personaddress_tbl}
) patAddrLink ON patAddrLink.PersonId = per.Id AND patAddrLink.rn = 1
LEFT JOIN {address_tbl} addr ON addr.Id = patAddrLink.AddressId
LEFT JOIN (
    SELECT PersonId, PhoneId,
           ROW_NUMBER() OVER (PARTITION BY PersonId ORDER BY Id) as rn
    FROM {personphonenumber_tbl}
) patPhoneLink ON patPhoneLink.PersonId = per.Id AND patPhoneLink.rn = 1
LEFT JOIN {phonenumber_tbl} pn ON pn.Id = patPhoneLink.PhoneId
LEFT JOIN (
    SELECT PayerId, AddressId,
           ROW_NUMBER() OVER (PARTITION BY PayerId ORDER BY Id) as rn
    FROM {payeraddress_tbl}
) payAddrLink ON payAddrLink.PayerId = pay.Id AND payAddrLink.rn = 1
LEFT JOIN {address_tbl} payAddr ON payAddr.Id = payAddrLink.AddressId
LEFT JOIN (
    SELECT PayerId, 
           MAX(CASE WHEN SandataServiceCodeId IS NOT NULL THEN 1 ELSE 0 END) as HasSandata,
           MAX(CASE WHEN CellTrakServiceCodeId IS NOT NULL THEN 1 ELSE 0 END) as HasCellTrak
    FROM {payerbillingcode_tbl}
    GROUP BY PayerId
) evv_agg ON evv_agg.PayerId = pay.Id
LEFT JOIN (
    SELECT PatientId, Icd9Cm,
           ROW_NUMBER() OVER (PARTITION BY PatientId ORDER BY Id) as rn
    FROM {icd_tbl}
    WHERE IsPrimary = true
) icd ON icd.PatientId = p.Id AND icd.rn = 1
LEFT JOIN (
    SELECT BillingLedgerId, 
           SUM(Amount) as TotalPayments
    FROM {payment_tbl}
    GROUP BY BillingLedgerId
) pay_agg ON pay_agg.BillingLedgerId = bl.Id
LEFT JOIN (
    SELECT BillingLedgerId, 
           SUM(Amount) as TotalAdjustments
    FROM {writeoff_tbl}
    GROUP BY BillingLedgerId
) wo_agg ON wo_agg.BillingLedgerId = bl.Id
WHERE bl.isActive='true'
""")
)

In [0]:
spark.sql(f"""
TRUNCATE TABLE {cubhub_charges}
""")

In [0]:
display(
spark.sql(f"""
INSERT INTO {cubhub_charges}
SELECT
    b.Id AS OfficeId,
    ab.Id AS ChargeId,
    bl.Id AS ClaimId,
    a.Id AS AssignmentId,
    pa.AuthorizationNumber AS AuthorizationNumber,
    bc.Code AS HcsPcsCode,
    pbc.RevenueCode AS RevenueCode,
    ROUND((UNIX_TIMESTAMP(ab.End) - UNIX_TIMESTAMP(ab.Start))/3600, 2) AS BilledHours,
    ROUND((UNIX_TIMESTAMP(ab.End) - UNIX_TIMESTAMP(ab.Start))/3600 * 4, 0) AS BilledUnits,  -- 4 units per hour (15 min units)
    1 AS BilledVisits,
    bld.Rate AS Rate,
    CASE bld.BillingRateType 
        WHEN 1 THEN 'Per Hour'
        WHEN 2 THEN 'Per Unit'
        WHEN 3 THEN 'Per Visit'
    END AS RateType,
    COALESCE(bld.DiscountAmount, 0.00) AS DiscountAmount,
    CASE WHEN bld.DiscountType IS NULL OR bld.DiscountType = 0 THEN 'None' ELSE CAST(bld.DiscountType AS STRING) END AS DiscountType,
    ROUND((UNIX_TIMESTAMP(ab.End) - UNIX_TIMESTAMP(ab.Start))/3600 * 4, 0) * bld.Rate / 4 AS CalculatedAmount,
    icd.Icd9Cm AS DiagnosisCode,
    bl.AmountBilled AS TotalBilled,
    bl.ClaimNumber AS ClaimNumber,
    b.ExternalId AS OfficeExternalId,
    DATE_FORMAT(DATE(ab.Start), 'yyyy-MM-dd') AS DateOfService,
    DATE_FORMAT(bl.DateBilled, 'yyyy-MM-dd') AS DateBilled,
    'PDN' AS Skill,
    fc.DisplayName AS FinancialClass,
    fc.Abbreviation AS FinancialClassAbbreviation,
    COALESCE(bcm1.Code, '') AS Modifier1,
    COALESCE(bcm2.Code, '') AS Modifier2,
    COALESCE(bcm3.Code, '') AS Modifier3,
    COALESCE(bcm4.Code, '') AS Modifier4,
    REPLACE(REPLACE(REPLACE(TRIM(SPLIT(bld.ServiceCodeString, '-')[0]), '\r', ''), '\n', ''), '\t', '') AS Discipline,
    bld.ServiceCodeString AS ServiceCodeDescription,
    CURRENT_DATE() AS LoadDate,
    CAST(CURRENT_DATE() AS STRING) AS FileName
FROM {billingledger_tbl} bl
JOIN {patientpayer_tbl} pp ON pp.Id = bl.PatientPayerId
JOIN {patient_tbl} p ON p.Id = pp.PatientId
JOIN {branch_tbl} b ON b.Id = p.BranchId
JOIN {payer_tbl} pay ON pay.Id = pp.PayerId
LEFT JOIN {financialclass_tbl} fc ON fc.Id = pay.FinancialClassId
-- Assignment billing (main granularity)
JOIN {assignment_tbl} a ON a.PatientId = p.Id
JOIN {assignmentbilling_tbl} ab 
    ON ab.AssignmentId = a.Id
    AND DATE(ab.start) >= to_date(split(DatesOfService, ' - ')[0], 'MM/dd/yyyy')
    AND DATE(ab.start) <= to_date(split(DatesOfService, ' - ')[1], 'MM/dd/yyyy')
    AND a.AssignedId IS NOT NULL
-- Link to billingledgerdetail via date
LEFT JOIN {billingledgerdetail_tbl} bld 
    ON bld.BillingLedgerId = bl.Id 
    AND DATE(bld.ServiceDate) = DATE(ab.Start)
-- Authorization based on date range
LEFT JOIN (
    SELECT PatientPayerId, AuthorizationNumber, AuthorizationStartDate, AuthorizationEndDate,
           ROW_NUMBER() OVER (PARTITION BY PatientPayerId ORDER BY AuthorizationStartDate DESC) as rn
    FROM {payerauthorization_tbl}
) pa ON pa.PatientPayerId = pp.Id 
    AND DATE(ab.Start) >= DATE(pa.AuthorizationStartDate) 
    AND DATE(ab.Start) <= DATE(pa.AuthorizationEndDate)
-- Diagnosis (pick J9611 - respiratory)
LEFT JOIN (
    SELECT PatientId, Icd9Cm,
           ROW_NUMBER() OVER (PARTITION BY PatientId ORDER BY CASE WHEN Icd9Cm LIKE 'J%' THEN 0 ELSE 1 END, Date DESC) as rn
    FROM {icd_tbl}
) icd ON icd.PatientId = p.Id AND icd.rn = 1
-- Billing Code (via assignmentbilling)
LEFT JOIN {payerauthorizationsegmentbillingcode_tbl} pasbc 
    ON pasbc.Id = ab.PayerAuthorizationSegmentBillingCodeId
LEFT JOIN {payerbillingcode_tbl} pbc ON pbc.Id = pasbc.PayerBillingCodeId
LEFT JOIN {billingcode_tbl} bc ON bc.Id = pbc.BillingCodeId
-- Modifiers
LEFT JOIN {payerbillingcodemodifier_tbl} pbcm1 
    ON pbcm1.PayerBillingCodeId = pbc.Id AND pbcm1.Position = 1
LEFT JOIN {billingcodemodifier_tbl} bcm1 
    ON bcm1.Id = pbcm1.ModifierId
LEFT JOIN {payerbillingcodemodifier_tbl} pbcm2 
    ON pbcm2.PayerBillingCodeId = pbc.Id AND pbcm2.Position = 2
LEFT JOIN {billingcodemodifier_tbl} bcm2 
    ON bcm2.Id = pbcm2.ModifierId
LEFT JOIN {payerbillingcodemodifier_tbl} pbcm3 
    ON pbcm3.PayerBillingCodeId = pbc.Id AND pbcm3.Position = 3
LEFT JOIN {billingcodemodifier_tbl} bcm3 
    ON bcm3.Id = pbcm3.ModifierId
LEFT JOIN {payerbillingcodemodifier_tbl} pbcm4 
    ON pbcm4.PayerBillingCodeId = pbc.Id AND pbcm4.Position = 4
LEFT JOIN {billingcodemodifier_tbl} bcm4 
    ON bcm4.Id = pbcm4.ModifierId
AND bl.isActive='true'
ORDER BY ab.Start
""")
)

In [0]:
spark.sql(f"""
TRUNCATE TABLE {cubhub_transactions}
""")

In [0]:
display(
spark.sql(f"""
INSERT INTO {cubhub_transactions}
-- Payments with paymentline (detail level)
SELECT DISTINCT
    b.Id AS OfficeId,
    bl.Id AS ClaimId,
    CASE pay.PaymentType
        WHEN 1 THEN 'Credit Card'
        WHEN 2 THEN 'EFT'
        WHEN 3 THEN 'Manual Adj'
        WHEN 5 THEN 'Refund'
        WHEN 7 THEN 'Transfer In'
        WHEN 8 THEN 'Transfer Out'
        WHEN 9 THEN 'Reversal'
        WHEN 10 THEN 'Credit Memo'
        WHEN 11 THEN 'CC Reversal'
        WHEN 12 THEN 'ACH'
        WHEN 13 THEN 'Write-On'
        ELSE CAST(pay.PaymentType AS STRING)
    END AS TransCode,
    pay.PaymentDetails AS TransDesc,
    CASE pay.PaymentType
        WHEN 1 THEN 'Payment'
        WHEN 2 THEN 'Payment'
        WHEN 12 THEN 'Payment'
        ELSE 'Adjustment'
    END AS TransType,
    DATE_FORMAT(pay.CreatedDate, 'MM/dd/yyyy') AS EntryDate,
    CAST(pay.Id AS STRING) AS PaymentId,
    pl.Amount AS TransAmt,
    NULL AS BatchId,
    DATE_FORMAT(pay.PaymentDate, 'MM/dd/yyyy') AS TransDate,
    pl.BillingLedgerDetailId,
    b.ExternalId AS OfficeExternalId,
    bl.ClaimNumber,
    CURRENT_DATE() AS Loaddate,
    CAST(CURRENT_DATE() AS STRING) AS FileName,
    NULL AS ReportingWeekEndingDate
FROM {billingledger_tbl} bl
JOIN {patientpayer_tbl} pp ON pp.Id = bl.PatientPayerId
JOIN {patient_tbl} p ON p.Id = pp.PatientId
JOIN {branch_tbl} b ON b.Id = p.BranchId
JOIN {payment_tbl} pay ON bl.Id = pay.BillingLedgerId
JOIN {paymentline_tbl} pl ON pl.PaymentId = pay.Id
WHERE bl.isActive='true'
UNION

-- Payments without paymentline (claim level)
SELECT DISTINCT
    b.Id AS OfficeId,
    bl.Id AS ClaimId,
    CASE pay.PaymentType
        WHEN 1 THEN 'Credit Card'
        WHEN 2 THEN 'EFT'
        WHEN 3 THEN 'Manual Adj'
        WHEN 5 THEN 'Refund'
        WHEN 7 THEN 'Transfer In'
        WHEN 8 THEN 'Transfer Out'
        WHEN 9 THEN 'Reversal'
        WHEN 10 THEN 'Credit Memo'
        WHEN 11 THEN 'CC Reversal'
        WHEN 12 THEN 'ACH'
        WHEN 13 THEN 'Write-On'
        ELSE CAST(pay.PaymentType AS STRING)
    END AS TransCode,
    pay.PaymentDetails AS TransDesc,
    CASE pay.PaymentType
        WHEN 1 THEN 'Payment'
        WHEN 2 THEN 'Payment'
        WHEN 12 THEN 'Payment'
        ELSE 'Adjustment'
    END AS TransType,
    DATE_FORMAT(pay.CreatedDate, 'MM/dd/yyyy') AS EntryDate,
    CAST(pay.Id  AS STRING) AS PaymentId,
    pay.Amount AS TransAmt,
    NULL AS BatchId,
    DATE_FORMAT(pay.PaymentDate, 'MM/dd/yyyy') AS TransDate,
    NULL AS BillingLedgerDetailId,
    b.ExternalId AS OfficeExternalId,
    bl.ClaimNumber,
    CURRENT_DATE() AS Loaddate,
    CAST(CURRENT_DATE() AS STRING) AS FileName,
    NULL AS ReportingWeekEndingDate
FROM {billingledger_tbl} bl
JOIN {patientpayer_tbl} pp ON pp.Id = bl.PatientPayerId
JOIN {patient_tbl} p ON p.Id = pp.PatientId
JOIN {branch_tbl} b ON b.Id = p.BranchId
JOIN {payment_tbl} pay ON bl.Id = pay.BillingLedgerId
WHERE NOT EXISTS (
    SELECT 1 FROM {paymentline_tbl} pl WHERE pl.PaymentId = pay.Id
)
AND bl.isActive='true'
UNION

-- Adjustments from writeoff (claim level)
SELECT DISTINCT
    b.Id AS OfficeId,
    bl.Id AS ClaimId,
    'Credit Memo' AS TransCode,
    wo.Notes AS TransDesc,
    'Adjustment' AS TransType,
    DATE_FORMAT(wo.CreatedDate, 'MM/dd/yyyy') AS EntryDate,
    NULL AS PaymentId,
    wo.Amount AS TransAmt,
    NULL AS BatchId,
    DATE_FORMAT(wo.DateBilled, 'MM/dd/yyyy') AS TransDate,
    NULL AS BillingLedgerDetailId,
    b.ExternalId AS OfficeExternalId,
    bl.ClaimNumber,
    CURRENT_DATE() AS Loaddate,
    CAST(CURRENT_DATE() AS STRING) AS FileName,
    NULL AS ReportingWeekEndingDate
FROM {billingledger_tbl} bl
JOIN {patientpayer_tbl} pp ON pp.Id = bl.PatientPayerId
JOIN {patient_tbl} p ON p.Id = pp.PatientId
JOIN {branch_tbl} b ON b.Id = p.BranchId
JOIN {writeoff_tbl} wo ON bl.Id = wo.BillingLedgerId
WHERE NOT EXISTS (
    SELECT 1 FROM {writeoffline_tbl} wol WHERE wol.WriteOffId = wo.Id
)
AND bl.isActive='true'
UNION

-- Adjustments from writeoffline (detail level)
SELECT DISTINCT
    b.Id AS OfficeId,
    bl.Id AS ClaimId,
    'Credit Memo' AS TransCode,
    wo.Notes AS TransDesc,
    'Adjustment' AS TransType,
    DATE_FORMAT(wo.CreatedDate, 'MM/dd/yyyy') AS EntryDate,
    NULL AS PaymentId,
    wol.Amount AS TransAmt,
    NULL AS BatchId,
    DATE_FORMAT(wo.DateBilled, 'MM/dd/yyyy') AS TransDate,
    wol.BillingLedgerDetailId,
    b.ExternalId AS OfficeExternalId,
    bl.ClaimNumber,
    CURRENT_DATE() AS Loaddate,
    CAST(CURRENT_DATE() AS STRING) AS FileName,
    NULL AS ReportingWeekEndingDate
FROM {billingledger_tbl} bl
JOIN {patientpayer_tbl} pp ON pp.Id = bl.PatientPayerId
JOIN {patient_tbl} p ON p.Id = pp.PatientId
JOIN {branch_tbl} b ON b.Id = p.BranchId
JOIN {writeoff_tbl} wo ON bl.Id = wo.BillingLedgerId
JOIN {writeoffline_tbl} wol ON wol.WriteOffId = wo.Id
WHERE bl.isActive='true'
""")
)